In [1]:
import os
print(os.listdir("data"))

['AI basics.pdf', 'Climate change (dehli gov).pdf', 'introduction-to-nutrition.pdf', 'Our-Solar-System-Book.pdf']


In [2]:
from langchain_community.document_loaders import PyPDFLoader
import os

docs = []

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join("data", file))
        docs.extend(loader.load())

print(f"Loaded {len(docs)} pages")

Loaded 207 pages


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}")


import random
random.shuffle(chunks)

Total chunks: 587


In [9]:
!pip install -U langchain-ollama

In [10]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [12]:
chunks = splitter.split_documents(docs)

import random
random.shuffle(chunks)

In [14]:
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="vectorstore"
)

vectordb.persist()

print("Vector DB created ✅")

Vector DB created ✅


C:\Users\diya\AppData\Local\Temp\ipykernel_52380\3625251326.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [19]:
retriever = vectordb.as_retriever(
    search_type="mmr",   # 🔥 key change
    search_kwargs={"k": 7, "fetch_k": 20}
)

In [20]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3")

In [21]:
def ask_question(query):
    docs = retriever.invoke(query)
    
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = f"""
Answer the question using the context below.

If the exact definition is not present, provide the best possible explanation based on the context.


Keep the answer concise (4–6 lines), clear, and professional.
Context:
{context}

Question:
{query}
"""
    
    response = llm.invoke(prompt)
    
    return response, docs

In [25]:
answer, sources = ask_question("Who is king of Mars?")

print("Answer:\n", answer)

print("\nSources:")
for s in sources:
    print(s.metadata)

Answer:
 Based on the provided context, there is no mention of Mars or any king related to it. The context appears to be about Artificial Intelligence (AI) and its capabilities. Therefore, I cannot provide an answer to this question using the given context.

Sources:
{'page': 28, 'page_label': '29', 'creator': 'Microsoft® PowerPoint® 2016', 'producer': 'www.ilovepdf.com', 'total_pages': 40, 'creationdate': '2020-07-01T05:13:10+00:00', 'moddate': '2020-07-01T05:13:11+00:00', 'source': 'data\\Climate change (dehli gov).pdf'}
{'producer': 'www.ilovepdf.com', 'page': 1, 'creationdate': '2020-07-01T05:13:10+00:00', 'total_pages': 40, 'creator': 'Microsoft® PowerPoint® 2016', 'page_label': '2', 'moddate': '2020-07-01T05:13:11+00:00', 'source': 'data\\Climate change (dehli gov).pdf'}
{'creationdate': '2024-03-06T16:39:02+00:00', 'source': 'data\\AI basics.pdf', 'page_label': '4', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'total_pages': 44, 'producer': 'Adobe PDF Library 17.